In [ ]:
!pip install pymongo pandas requests python-dotenv

In [ ]:
import os
import time
import logging
from dataclasses import dataclass
from datetime import datetime, timedelta, timezone

import requests
import pandas as pd

from pymongo import MongoClient, UpdateOne

In [ ]:
@dataclass(frozen=True)
class Settings:
    mongodb_uri: str = os.getenv("MONGODB_URI", "")
    mongo_db_name: str = os.getenv("MONGO_DB_NAME", "masters_project")
    odds_api_key: str = os.getenv("ODDS_API_KEY", "")
    news_api_key: str = os.getenv("NEWS_API_KEY", "")
    tournament_name: str = os.getenv("TOURNAMENT_NAME", "Masters Tournament Winner")
    sport_key: str = os.getenv("SPORT_KEY", "golf_masters_tournament_winner")
    region: str = os.getenv("ODDS_REGION", "us")
    market: str = os.getenv("ODDS_MARKET", "outrights")
    news_query_template: str = os.getenv(
        "NEWS_QUERY_TEMPLATE",
        '"{player}" AND (Masters OR Augusta OR golf)'
    )

SETTINGS = Settings()

logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s")

# MongoDB helpers

In [ ]:
def get_client():
    if not SETTINGS.mongodb_uri:
        raise ValueError("MONGODB_URI is missing.")
    return MongoClient(SETTINGS.mongodb_uri)

def get_db():
    return get_client()[SETTINGS.mongo_db_name]

def get_collection(name: str):
    return get_db()[name]

def ensure_indexes():
    db = get_db()

    db["odds_snapshots"].create_index(
        [("snapshot_time_utc", 1), ("player_name", 1)],
        unique=True
    )

    db["odds_snapshots"].create_index(
        [("player_name", 1), ("snapshot_time_utc", 1)]
    )

    logging.info("Indexes created successfully.")

# Helper functions

In [ ]:
def safe_float(value):
    try:
        return float(value)
    except (TypeError, ValueError):
        return None

def american_to_implied_prob(price):
    if price is None:
        return None
    if price > 0:
        return 100 / (price + 100)
    return abs(price) / (abs(price) + 100)

def generate_snapshot_times(start_iso: str, end_iso: str, step_minutes: int = 15):
    start_dt = datetime.fromisoformat(start_iso.replace("Z", "+00:00"))
    end_dt = datetime.fromisoformat(end_iso.replace("Z", "+00:00"))

    timestamps = []
    current = start_dt

    while current <= end_dt:
        timestamps.append(current.isoformat().replace("+00:00", "Z"))
        current += timedelta(minutes=step_minutes)

    return timestamps

# Fetch odds from The Odds API

In [ ]:
HISTORICAL_ODDS_URL = "https://api.the-odds-api.com/v4/historical/sports/{sport_key}/odds"

def fetch_historical_odds_payload(snapshot_date: str):
    if not SETTINGS.odds_api_key:
        raise ValueError("ODDS_API_KEY is missing.")

    params = {
        "apiKey": SETTINGS.odds_api_key,
        "regions": SETTINGS.region,
        "markets": SETTINGS.market,
        "oddsFormat": "american",
        "dateFormat": "iso",
        "date": snapshot_date,
    }

    response = requests.get(
        HISTORICAL_ODDS_URL.format(sport_key=SETTINGS.sport_key),
        params=params,
        timeout=30,
    )

    response.raise_for_status()
    return response.json(), response.headers

# Build and save odds documents

In [ ]:
def build_historical_odds_documents(payload, requested_snapshot_time: str):
    snapshot_timestamp = payload.get("timestamp")
    events = payload.get("data", [])
    documents = []

    for event in events:
        event_id = event.get("id")
        event_name = event.get("sport_title", SETTINGS.tournament_name)
        commence_time = event.get("commence_time")
        bookmakers = event.get("bookmakers", [])

        player_prices = {}

        for bookmaker in bookmakers:
            markets = bookmaker.get("markets", [])
            for market in markets:
                outcomes = market.get("outcomes", [])
                for outcome in outcomes:
                    player_name = outcome.get("name")
                    price = safe_float(outcome.get("price"))

                    if not player_name or price is None:
                        continue

                    player_prices.setdefault(player_name, []).append(price)

        for player_name, prices in player_prices.items():
            avg_american_price = sum(prices) / len(prices)
            raw_implied_prob = american_to_implied_prob(avg_american_price)

            doc = {
                "snapshot_time_utc": snapshot_timestamp,
                "requested_snapshot_time_utc": requested_snapshot_time,
                "tournament": SETTINGS.tournament_name,
                "event_id": event_id,
                "event_name": event_name,
                "commence_time": commence_time,
                "player_name": player_name,
                "odds": {
                    "average_american_price": avg_american_price,
                    "raw_implied_prob": raw_implied_prob,
                    "sportsbook_quote_count": len(prices),
                },
                "source": {
                    "provider": "The Odds API",
                    "mode": "historical",
                    "sport_key": SETTINGS.sport_key,
                    "market": SETTINGS.market,
                },
            }

            documents.append(doc)

    total_prob = sum(
        doc["odds"]["raw_implied_prob"]
        for doc in documents
        if doc["odds"]["raw_implied_prob"] is not None
    )

    if total_prob > 0:
        for doc in documents:
            raw_prob = doc["odds"]["raw_implied_prob"]
            doc["odds"]["normalized_implied_prob"] = (
                raw_prob / total_prob if raw_prob is not None else None
            )

    return documents

# Upsert odds documents



In [ ]:
def upsert_odds_documents(documents):
    if not documents:
        return 0

    collection = get_collection("odds_snapshots")
    operations = []

    for doc in documents:
        operations.append(
            UpdateOne(
                {
                    "snapshot_time_utc": doc["snapshot_time_utc"],
                    "player_name": doc["player_name"],
                },
                {"$set": doc},
                upsert=True,
            )
        )

    result = collection.bulk_write(operations, ordered=False)
    return result.upserted_count + result.modified_count

# Time Range

In [ ]:
START_SNAPSHOT = "2026-04-09T00:00:00Z"
END_SNAPSHOT   = "2026-04-12T19:00:00Z"
STEP_MINUTES   = 15

snapshot_times = generate_snapshot_times(
    start_iso=START_SNAPSHOT,
    end_iso=END_SNAPSHOT,
    step_minutes=STEP_MINUTES
)

print("Total snapshots to request:", len(snapshot_times))
print(snapshot_times[:5], "...")
print(snapshot_times[-5:])

Total snapshots to request: 365
['2026-04-09T00:00:00Z', '2026-04-09T00:15:00Z', '2026-04-09T00:30:00Z', '2026-04-09T00:45:00Z', '2026-04-09T01:00:00Z'] ...
['2026-04-12T18:00:00Z', '2026-04-12T18:15:00Z', '2026-04-12T18:30:00Z', '2026-04-12T18:45:00Z', '2026-04-12T19:00:00Z']


# Run Collection Group

In [ ]:
ensure_indexes()

all_requested = 0
all_events = 0
all_documents = 0
all_writes = 0

for i, snapshot_time in enumerate(snapshot_times, start=1):
    try:
        payload, headers = fetch_historical_odds_payload(snapshot_time)
        docs = build_historical_odds_documents(payload, snapshot_time)
        writes = upsert_odds_documents(docs)

        all_requested += 1
        all_events += len(payload.get("data", []))
        all_documents += len(docs)
        all_writes += writes

        print(
            f"[{i}/{len(snapshot_times)}] "
            f"{snapshot_time} | "
            f"events={len(payload.get('data', []))} | "
            f"docs={len(docs)} | "
            f"writes={writes} | "
            f"remaining={headers.get('x-requests-remaining')}"
        )

        time.sleep(0.2)

    except Exception as exc:
        print(f"[{i}/{len(snapshot_times)}] FAILED at {snapshot_time}: {exc}")

print("\nDONE")
print("Snapshots requested:", all_requested)
print("Total events returned:", all_events)
print("Total odds documents built:", all_documents)
print("Total Mongo writes:", all_writes)

[1/365] 2026-04-09T00:00:00Z | events=1 | docs=92 | writes=92 | remaining=15299
[2/365] 2026-04-09T00:15:00Z | events=1 | docs=92 | writes=92 | remaining=15289
[3/365] 2026-04-09T00:30:00Z | events=1 | docs=92 | writes=92 | remaining=15279
[4/365] 2026-04-09T00:45:00Z | events=1 | docs=92 | writes=92 | remaining=15269
[5/365] 2026-04-09T01:00:00Z | events=1 | docs=92 | writes=92 | remaining=15259
[6/365] 2026-04-09T01:15:00Z | events=1 | docs=92 | writes=92 | remaining=15249
[7/365] 2026-04-09T01:30:00Z | events=1 | docs=92 | writes=92 | remaining=15239
[8/365] 2026-04-09T01:45:00Z | events=1 | docs=92 | writes=92 | remaining=15229
[9/365] 2026-04-09T02:00:00Z | events=1 | docs=92 | writes=92 | remaining=15219
[10/365] 2026-04-09T02:15:00Z | events=1 | docs=92 | writes=92 | remaining=15209
[11/365] 2026-04-09T02:30:00Z | events=1 | docs=92 | writes=92 | remaining=15199
[12/365] 2026-04-09T02:45:00Z | events=1 | docs=92 | writes=92 | remaining=15189
[13/365] 2026-04-09T03:00:00Z | event

In [ ]:
preview_docs = list(
    get_collection("odds_snapshots").find({}, {"_id": 0}).sort("snapshot_time_utc", 1).limit(10)
)

pd.json_normalize(preview_docs)

,player_name,snapshot_time_utc,commence_time,event_id,event_name,requested_snapshot_time_utc,tournament,odds.average_american_price,odds.raw_implied_prob,odds.sportsbook_quote_count,odds.normalized_implied_prob,source.provider,source.mode,source.sport_key,source.market
0,Aaron Rai,2026-04-08T23:55:38Z,2026-04-09T11:40:00Z,b709a17e8bec948af575f4534c598144,Masters Tournament Winner,2026-04-09T00:00:00Z,Masters Tournament Winner,23333.333333,0.004267,6,0.003171,The Odds API,historical,golf_masters_tournament_winner,outrights
1,Adam Scott,2026-04-08T23:55:38Z,2026-04-09T11:40:00Z,b709a17e8bec948af575f4534c598144,Masters Tournament Winner,2026-04-09T00:00:00Z,Masters Tournament Winner,7450.000000,0.013245,6,0.009841,The Odds API,historical,golf_masters_tournament_winner,outrights
2,Akshay Bhatia,2026-04-08T23:55:38Z,2026-04-09T11:40:00Z,b709a17e8bec948af575f4534c598144,Masters Tournament Winner,2026-04-09T00:00:00Z,Masters Tournament Winner,5766.666667,0.017045,6,0.012664,The Odds API,historical,golf_masters_tournament_winner,outrights
3,Aldrich Potgieter,2026-04-08T23:55:38Z,2026-04-09T11:40:00Z,b709a17e8bec948af575f4534c598144,Masters Tournament Winner,2026-04-09T00:00:00Z,Masters Tournament Winner,25166.666667,0.003958,6,0.002940,The Odds API,historical,golf_masters_tournament_winner,outrights
4,Alexander Noren,2026-04-08T23:55:38Z,2026-04-09T11:40:00Z,b709a17e8bec948af575f4534c598144,Masters Tournament Winner,2026-04-09T00:00:00Z,Masters Tournament Winner,19333.333333,0.005146,6,0.003823,The Odds API,historical,golf_masters_tournament_winner,outrights
5,Andrew Novak,2026-04-08T23:55:38Z,2026-04-09T11:40:00Z,b709a17e8bec948af575f4534c598144,Masters Tournament Winner,2026-04-09T00:00:00Z,Masters Tournament Winner,34833.333333,0.002863,6,0.002127,The Odds API,historical,golf_masters_tournament_winner,outrights
6,Angel Cabrera,2026-04-08T23:55:38Z,2026-04-09T11:40:00Z,b709a17e8bec948af575f4534c598144,Masters Tournament Winner,2026-04-09T00:00:00Z,Masters Tournament Winner,183333.333333,0.000545,6,0.000405,The Odds API,historical,golf_masters_tournament_winner,outrights
7,Ben Griffin,2026-04-08T23:55:38Z,2026-04-09T11:40:00Z,b709a17e8bec948af575f4534c598144,Masters Tournament Winner,2026-04-09T00:00:00Z,Masters Tournament Winner,14333.333333,0.006928,6,0.005148,The Odds API,historical,golf_masters_tournament_winner,outrights
8,Brandon Holtz,2026-04-08T23:55:38Z,2026-04-09T11:40:00Z,b709a17e8bec948af575f4534c598144,Masters Tournament Winner,2026-04-09T00:00:00Z,Masters Tournament Winner,200000.000000,0.000500,6,0.000371,The Odds API,historical,golf_masters_tournament_winner,outrights
9,Brian Campbell,2026-04-08T23:55:38Z,2026-04-09T11:40:00Z,b709a17e8bec948af575f4534c598144,Masters Tournament Winner,2026-04-09T00:00:00Z,Masters Tournament Winner,150000.000000,0.000666,6,0.000495,The Odds API,historical,golf_masters_tournament_winner,outrights


In [ ]:
odds_docs = list(
    get_collection("odds_snapshots").find({}, {"_id": 0})
)

odds_df = pd.json_normalize(odds_docs)
print(odds_df.shape)
odds_df.head()

(26157, 15)


,player_name,snapshot_time_utc,commence_time,event_id,event_name,requested_snapshot_time_utc,tournament,odds.average_american_price,odds.raw_implied_prob,odds.sportsbook_quote_count,odds.normalized_implied_prob,source.provider,source.mode,source.sport_key,source.market
0,Scottie Scheffler,2026-04-08T23:55:38Z,2026-04-09T11:40:00Z,b709a17e8bec948af575f4534c598144,Masters Tournament Winner,2026-04-09T00:00:00Z,Masters Tournament Winner,571.666667,0.148883,6,0.110615,The Odds API,historical,golf_masters_tournament_winner,outrights
1,Bryson DeChambeau,2026-04-08T23:55:38Z,2026-04-09T11:40:00Z,b709a17e8bec948af575f4534c598144,Masters Tournament Winner,2026-04-09T00:00:00Z,Masters Tournament Winner,1091.666667,0.083916,6,0.062346,The Odds API,historical,golf_masters_tournament_winner,outrights
2,Jon Rahm,2026-04-08T23:55:38Z,2026-04-09T11:40:00Z,b709a17e8bec948af575f4534c598144,Masters Tournament Winner,2026-04-09T00:00:00Z,Masters Tournament Winner,1033.333333,0.088235,6,0.065555,The Odds API,historical,golf_masters_tournament_winner,outrights
3,Ludvig Aberg,2026-04-08T23:55:38Z,2026-04-09T11:40:00Z,b709a17e8bec948af575f4534c598144,Masters Tournament Winner,2026-04-09T00:00:00Z,Masters Tournament Winner,1583.333333,0.059406,6,0.044136,The Odds API,historical,golf_masters_tournament_winner,outrights
4,Rory McIlroy,2026-04-08T23:55:38Z,2026-04-09T11:40:00Z,b709a17e8bec948af575f4534c598144,Masters Tournament Winner,2026-04-09T00:00:00Z,Masters Tournament Winner,1320.833333,0.070381,6,0.052291,The Odds API,historical,golf_masters_tournament_winner,outrights


# Compute Odds Change

In [ ]:
odds_df["snapshot_time_utc"] = pd.to_datetime(odds_df["snapshot_time_utc"], utc=True)

odds_df = odds_df.sort_values(["player_name", "snapshot_time_utc"]).reset_index(drop=True)

odds_df["odds_change_avg_american_price"] = (
    odds_df.groupby("player_name")["odds.average_american_price"].diff()
)

odds_df["odds_change_raw_implied_prob"] = (
    odds_df.groupby("player_name")["odds.raw_implied_prob"].diff()
)

odds_df["odds_change_normalized_implied_prob"] = (
    odds_df.groupby("player_name")["odds.normalized_implied_prob"].diff()
)

odds_df.head(20)

,player_name,snapshot_time_utc,commence_time,event_id,event_name,requested_snapshot_time_utc,tournament,odds.average_american_price,odds.raw_implied_prob,odds.sportsbook_quote_count,odds.normalized_implied_prob,source.provider,source.mode,source.sport_key,source.market,odds_change_avg_american_price,odds_change_raw_implied_prob,odds_change_normalized_implied_prob
0,Aaron Rai,2026-04-08 23:55:38+00:00,2026-04-09T11:40:00Z,b709a17e8bec948af575f4534c598144,Masters Tournament Winner,2026-04-09T00:00:00Z,Masters Tournament Winner,23333.333333,0.004267,6,0.003171,The Odds API,historical,golf_masters_tournament_winner,outrights,NaN,NaN,NaN
1,Aaron Rai,2026-04-09 00:10:38+00:00,2026-04-09T11:40:00Z,b709a17e8bec948af575f4534c598144,Masters Tournament Winner,2026-04-09T00:15:00Z,Masters Tournament Winner,23333.333333,0.004267,6,0.001841,The Odds API,historical,golf_masters_tournament_winner,outrights,0.000000,0.000000,-1.329697e-03
2,Aaron Rai,2026-04-09 00:25:38+00:00,2026-04-09T11:40:00Z,b709a17e8bec948af575f4534c598144,Masters Tournament Winner,2026-04-09T00:30:00Z,Masters Tournament Winner,23333.333333,0.004267,6,0.003169,The Odds API,historical,golf_masters_tournament_winner,outrights,0.000000,0.000000,1.327928e-03
3,Aaron Rai,2026-04-09 00:40:38+00:00,2026-04-09T11:40:00Z,b709a17e8bec948af575f4534c598144,Masters Tournament Winner,2026-04-09T00:45:00Z,Masters Tournament Winner,23333.333333,0.004267,6,0.003169,The Odds API,historical,golf_masters_tournament_winner,outrights,0.000000,0.000000,0.000000e+00
4,Aaron Rai,2026-04-09 00:55:38+00:00,2026-04-09T11:40:00Z,b709a17e8bec948af575f4534c598144,Masters Tournament Winner,2026-04-09T01:00:00Z,Masters Tournament Winner,23333.333333,0.004267,6,0.003169,The Odds API,historical,golf_masters_tournament_winner,outrights,0.000000,0.000000,0.000000e+00
5,Aaron Rai,2026-04-09 01:10:38+00:00,2026-04-09T11:40:00Z,b709a17e8bec948af575f4534c598144,Masters Tournament Winner,2026-04-09T01:15:00Z,Masters Tournament Winner,23333.333333,0.004267,6,0.003168,The Odds API,historical,golf_masters_tournament_winner,outrights,0.000000,0.000000,-9.232482e-07
6,Aaron Rai,2026-04-09 01:25:38+00:00,2026-04-09T11:40:00Z,b709a17e8bec948af575f4534c598144,Masters Tournament Winner,2026-04-09T01:30:00Z,Masters Tournament Winner,23000.000000,0.004329,5,0.003229,The Odds API,historical,golf_masters_tournament_winner,outrights,-333.333333,0.000062,6.084433e-05
7,Aaron Rai,2026-04-09 01:40:38+00:00,2026-04-09T11:40:00Z,b709a17e8bec948af575f4534c598144,Masters Tournament Winner,2026-04-09T01:45:00Z,Masters Tournament Winner,23333.333333,0.004267,6,0.003168,The Odds API,historical,golf_masters_tournament_winner,outrights,333.333333,-0.000062,-6.084433e-05
8,Aaron Rai,2026-04-09 01:55:37+00:00,2026-04-09T11:40:00Z,b709a17e8bec948af575f4534c598144,Masters Tournament Winner,2026-04-09T02:00:00Z,Masters Tournament Winner,23333.333333,0.004267,6,0.003178,The Odds API,historical,golf_masters_tournament_winner,outrights,0.000000,0.000000,1.000357e-05
9,Aaron Rai,2026-04-09 02:10:38+00:00,2026-04-09T11:40:00Z,b709a17e8bec948af575f4534c598144,Masters Tournament Winner,2026-04-09T02:15:00Z,Masters Tournament Winner,23333.333333,0.004267,6,0.003177,The Odds API,historical,golf_masters_tournament_winner,outrights,0.000000,0.000000,-6.577872e-07


In [ ]:
def write_odds_changes_back_to_mongo(df: pd.DataFrame):
    collection = get_collection("odds_snapshots")
    operations = []

    for _, row in df.iterrows():
        operations.append(
            UpdateOne(
                {
                    "snapshot_time_utc": row["snapshot_time_utc"].isoformat().replace("+00:00", "Z"),
                    "player_name": row["player_name"],
                },
                {
                    "$set": {
                        "odds_change": {
                            "average_american_price": (
                                None if pd.isna(row["odds_change_avg_american_price"])
                                else float(row["odds_change_avg_american_price"])
                            ),
                            "raw_implied_prob": (
                                None if pd.isna(row["odds_change_raw_implied_prob"])
                                else float(row["odds_change_raw_implied_prob"])
                            ),
                            "normalized_implied_prob": (
                                None if pd.isna(row["odds_change_normalized_implied_prob"])
                                else float(row["odds_change_normalized_implied_prob"])
                            ),
                        }
                    }
                },
                upsert=False,
            )
        )

    if not operations:
        return 0

    result = collection.bulk_write(operations, ordered=False)
    return result.modified_count

In [ ]:
modified_count = write_odds_changes_back_to_mongo(odds_df)
print("Documents updated with odds_change:", modified_count)

Documents updated with odds_change: 26157


In [ ]:
preview_changed_docs = list(
    get_collection("odds_snapshots").find(
        {},
        {
            "_id": 0,
            "snapshot_time_utc": 1,
            "player_name": 1,
            "odds.average_american_price": 1,
            "odds.raw_implied_prob": 1,
            "odds.normalized_implied_prob": 1,
            "odds_change": 1,
        }
    ).sort([("player_name", 1), ("snapshot_time_utc", 1)]).limit(20)
)

pd.json_normalize(preview_changed_docs)

,player_name,snapshot_time_utc,odds.average_american_price,odds.raw_implied_prob,odds.normalized_implied_prob,odds_change.average_american_price,odds_change.raw_implied_prob,odds_change.normalized_implied_prob
0,Aaron Rai,2026-04-08T23:55:38Z,23333.333333,0.004267,0.003171,NaN,NaN,NaN
1,Aaron Rai,2026-04-09T00:10:38Z,23333.333333,0.004267,0.001841,0.000000,0.000000,-1.329697e-03
2,Aaron Rai,2026-04-09T00:25:38Z,23333.333333,0.004267,0.003169,0.000000,0.000000,1.327928e-03
3,Aaron Rai,2026-04-09T00:40:38Z,23333.333333,0.004267,0.003169,0.000000,0.000000,0.000000e+00
4,Aaron Rai,2026-04-09T00:55:38Z,23333.333333,0.004267,0.003169,0.000000,0.000000,0.000000e+00
5,Aaron Rai,2026-04-09T01:10:38Z,23333.333333,0.004267,0.003168,0.000000,0.000000,-9.232482e-07
6,Aaron Rai,2026-04-09T01:25:38Z,23000.000000,0.004329,0.003229,-333.333333,0.000062,6.084433e-05
7,Aaron Rai,2026-04-09T01:40:38Z,23333.333333,0.004267,0.003168,333.333333,-0.000062,-6.084433e-05
8,Aaron Rai,2026-04-09T01:55:37Z,23333.333333,0.004267,0.003178,0.000000,0.000000,1.000357e-05
9,Aaron Rai,2026-04-09T02:10:38Z,23333.333333,0.004267,0.003177,0.000000,0.000000,-6.577872e-07


# Read player list from your stored odds data

In [ ]:
players = sorted(
    get_collection("odds_snapshots").distinct("player_name")
)

print("Total players:", len(players))
print(players[:20])

Total players: 92
['Aaron Rai', 'Adam Scott', 'Akshay Bhatia', 'Aldrich Potgieter', 'Alexander Noren', 'Andrew Novak', 'Angel Cabrera', 'Ben Griffin', 'Brandon Holtz', 'Brian Campbell', 'Brian Harman', 'Brooks Koepka', 'Bryson DeChambeau', 'Bubba Watson', 'Cameron Smith', 'Cameron Young', 'Carlos Ortiz', 'Casey Jarvis', 'Charl Schwartzel', 'Christopher Gotterup']


# News settings

In [ ]:
NEWS_FROM = "2026-04-09T00:00:00"
NEWS_TO   = "2026-04-12T19:59:59"
NEWS_PAGE_SIZE = 100

# Create MongoDB indexes for news

In [ ]:
def ensure_news_indexes():
    db = get_db()

    db["news_articles"].create_index(
        [("url", 1)],
        unique=True,
        sparse=True
    )

    db["news_articles"].create_index(
        [("player_name", 1), ("published_at", 1)]
    )

    db["news_articles"].create_index(
        [("published_at", 1)]
    )

    logging.info("News indexes created successfully.")

# NewsAPI fetch function

In [ ]:
NEWS_API_URL = "https://newsapi.org/v2/everything"

def fetch_news_page_for_player(player_name, from_dt, to_dt, page=1, page_size=100):
    if not os.getenv("NEWS_API_KEY"):
        raise ValueError("NEWS_API_KEY is missing.")

    query = f'"{player_name}" AND (Masters OR Augusta OR golf)'

    params = {
        "q": query,
        "from": from_dt,
        "to": to_dt,
        "language": "en",
        "sortBy": "publishedAt",
        "pageSize": page_size,
        "page": page,
        "apiKey": os.getenv("NEWS_API_KEY"),
    }

    response = requests.get(NEWS_API_URL, params=params, timeout=30)
    response.raise_for_status()
    return response.json()

# Pull all pages for one player

In [ ]:
def fetch_all_news_for_player(player_name, from_dt, to_dt, page_size=100, max_pages=20):
    all_articles = []
    total_results = None

    for page in range(1, max_pages + 1):
        payload = fetch_news_page_for_player(
            player_name=player_name,
            from_dt=from_dt,
            to_dt=to_dt,
            page=page,
            page_size=page_size,
        )

        articles = payload.get("articles", [])
        if total_results is None:
            total_results = payload.get("totalResults", 0)

        all_articles.extend(articles)

        if len(articles) < page_size:
            break

    return all_articles

# Simple sentiment helper

In [ ]:
def simple_sentiment_score(text: str) -> float:
    positive_words = {"win", "strong", "favorite", "improved", "great", "top", "best"}
    negative_words = {"injury", "withdraw", "struggle", "poor", "bad", "miss", "doubt"}

    tokens = [token.strip(".,:;!?()[]{}\"'").lower() for token in text.split()]
    if not tokens:
        return 0.0

    score = sum(1 for token in tokens if token in positive_words) - sum(
        1 for token in tokens if token in negative_words
    )
    return score / len(tokens)

# Build news article documents

In [ ]:
def build_article_documents(player_name, articles, from_dt, to_dt):
    retrieved_at = datetime.now(timezone.utc).isoformat()
    documents = []

    for article in articles:
        title = article.get("title") or ""
        description = article.get("description") or ""
        content = article.get("content") or ""
        content_for_scoring = f"{title} {description} {content}".strip()

        url = article.get("url")
        published_at = article.get("publishedAt")

        fallback_key = f"{player_name}::{title}::{published_at}"

        doc = {
            "retrieved_at_utc": retrieved_at,
            "query_window": {
                "from": from_dt,
                "to": to_dt,
            },
            "tournament": SETTINGS.tournament_name,
            "player_name": player_name,
            "title": title,
            "description": description,
            "content": content,
            "author": article.get("author"),
            "published_at": published_at,
            "url": url,
            "url_to_image": article.get("urlToImage"),
            "source_name": (article.get("source") or {}).get("name"),
            "source_id": (article.get("source") or {}).get("id"),
            "sentiment": {
                "simple_score": simple_sentiment_score(content_for_scoring)
            },
            "source": {
                "provider": "NewsAPI",
                "endpoint": "everything",
                "query": f'"{player_name}" AND (Masters OR Augusta OR golf)',
            },
            "_fallback_key": fallback_key,
        }

        documents.append(doc)

    return documents

# Upsert news articles into MongoDB

In [ ]:
def upsert_news_documents(documents):
    if not documents:
        return 0

    collection = get_collection("news_articles")
    operations = []

    for doc in documents:
        if doc.get("url"):
            selector = {"url": doc["url"]}
        else:
            selector = {"_fallback_key": doc["_fallback_key"]}

        operations.append(
            UpdateOne(
                selector,
                {"$set": doc},
                upsert=True,
            )
        )

    result = collection.bulk_write(operations, ordered=False)
    return result.upserted_count + result.modified_count

# Run the historical news backfill

In [ ]:
ensure_news_indexes()

total_articles = 0
total_writes = 0

for i, player_name in enumerate(players, start=1):
    try:
        articles = fetch_all_news_for_player(
            player_name=player_name,
            from_dt=NEWS_FROM,
            to_dt=NEWS_TO,
            page_size=NEWS_PAGE_SIZE,
            max_pages=10,
        )

        docs = build_article_documents(
            player_name=player_name,
            articles=articles,
            from_dt=NEWS_FROM,
            to_dt=NEWS_TO,
        )

        writes = upsert_news_documents(docs)

        total_articles += len(docs)
        total_writes += writes

        print(f"[{i}/{len(players)}] {player_name} | articles={len(docs)} | writes={writes}")

    except Exception as exc:
        print(f"[{i}/{len(players)}] FAILED for {player_name}: {exc}")

print("\nDONE")
print("Total article docs:", total_articles)
print("Total writes:", total_writes)

[1/92] Aaron Rai | articles=43 | writes=43
[2/92] Adam Scott | articles=53 | writes=53
[3/92] Akshay Bhatia | articles=31 | writes=31
[4/92] Aldrich Potgieter | articles=19 | writes=19
[5/92] Alexander Noren | articles=0 | writes=0
[6/92] Andrew Novak | articles=19 | writes=19
[7/92] Angel Cabrera | articles=23 | writes=23
[8/92] Ben Griffin | articles=31 | writes=31
[9/92] Brandon Holtz | articles=28 | writes=28
[10/92] Brian Campbell | articles=28 | writes=28
[11/92] Brian Harman | articles=32 | writes=32
[12/92] Brooks Koepka | articles=70 | writes=70
[13/92] Bryson DeChambeau | articles=97 | writes=97
[14/92] Bubba Watson | articles=45 | writes=45
[15/92] Cameron Smith | articles=38 | writes=38
[16/92] Cameron Young | articles=96 | writes=96
[17/92] Carlos Ortiz | articles=16 | writes=16
[18/92] Casey Jarvis | articles=11 | writes=11
[19/92] Charl Schwartzel | articles=42 | writes=42
[20/92] Christopher Gotterup | articles=0 | writes=0
[21/92] Collin Morikawa | articles=53 | writes

# Preview stored news documents

In [ ]:
news_preview = list(
    get_collection("news_articles").find({}, {"_id": 0}).limit(10)
)

pd.json_normalize(news_preview)

,url,_fallback_key,author,content,description,player_name,published_at,retrieved_at_utc,source_id,source_name,title,tournament,url_to_image,query_window.from,query_window.to,sentiment.simple_score,source.provider,source.endpoint,source.query
0,http://robbreport.com/lifestyle/sports-leisure...,Shane Lowry::The Final Round of the Masters Co...,Viju Mathew,The stroll to the 11th tee box was a subdued s...,"For the Masters, past champion Bernhard Langer...",Shane Lowry,2026-04-12T13:00:00Z,2026-04-28T21:54:01.364710+00:00,None,Robb Report,The Final Round of the Masters Could Be as Thr...,Masters Tournament Winner,https://robbreport.com/wp-content/uploads/2026...,2026-04-09T00:00:00,2026-04-12T19:59:59,0.000000,NewsAPI,everything,"""Shane Lowry"" AND (Masters OR Augusta OR golf)"
1,http://robbreport.com/lifestyle/sports-leisure...,Shane Lowry::Why the Final Round of the Master...,Viju Mathew,The stroll to the 11th tee box was a subdued s...,"For the Masters, past champion Bernhard Langer...",Shane Lowry,2026-04-12T13:00:00Z,2026-04-28T21:54:01.364710+00:00,None,Robb Report,Why the Final Round of the Masters Could Be as...,Masters Tournament Winner,https://robbreport.com/wp-content/uploads/2026...,2026-04-09T00:00:00,2026-04-12T19:59:59,0.000000,NewsAPI,everything,"""Shane Lowry"" AND (Masters OR Augusta OR golf)"
2,https://www.thebiglead.com/2026-masters-sunday...,Viktor Hovland::2026 Masters Sunday tee times ...,Josh Sanchez,The 2026 Masters Tournament wraps up on Sunday...,The 2026 Masters Tournament wraps up on Sunday...,Viktor Hovland,2026-04-12T12:32:10Z,2026-04-28T21:54:06.584930+00:00,None,The Big Lead,2026 Masters Sunday tee times & pairings for R...,Masters Tournament Winner,https://s.yimg.com/os/en/the_big_lead_articles...,2026-04-09T00:00:00,2026-04-12T19:59:59,0.000000,NewsAPI,everything,"""Viktor Hovland"" AND (Masters OR Augusta OR golf)"
3,https://www.rediff.com/sports/report/aaron-rai...,"Shane Lowry::Masters: Rai's Round 3 struggles,...",sports@rediff.co.in (Rediff Sports Desk),Indo-British golfer Aaron Rai faced challenges...,Indo-British golfer Aaron Rai dropped to tied ...,Shane Lowry,2026-04-12T12:08:16Z,2026-04-28T21:54:01.364710+00:00,None,Rediff.com,"Masters: Rai's Round 3 struggles, Young and Mc...",Masters Tournament Winner,https://im.rediff.com/1200-630/sports/2024/jun...,2026-04-09T00:00:00,2026-04-12T19:59:59,0.013699,NewsAPI,everything,"""Shane Lowry"" AND (Masters OR Augusta OR golf)"
4,https://nypost.com/2026/04/12/sports/how-to-wa...,Xander Schauffele::How to watch the final roun...,Angela Tricarico,New York Post may be compensated and/or receiv...,It all comes down to this round.,Xander Schauffele,2026-04-12T10:00:00Z,2026-04-28T21:54:08.137868+00:00,None,New York Post,How to watch the final round of The Masters 20...,Masters Tournament Winner,https://nypost.com/wp-content/uploads/sites/2/...,2026-04-09T00:00:00,2026-04-12T19:59:59,0.000000,NewsAPI,everything,"""Xander Schauffele"" AND (Masters OR Augusta OR..."
5,https://golf.com/news/he-missed-masters-cut-be...,Tommy Fleetwood::He missed the Masters cut. Sa...,Nick Piastowski,"AUGUSTA, Ga. Is that Dwyane Wade watching his ...",Brandon Holtz missed the Masters cut on Friday.,Tommy Fleetwood,2026-04-12T06:38:59Z,2026-04-28T21:54:03.828992+00:00,None,Golf.com,"He missed the Masters cut. Saturday, we drank ...",Masters Tournament Winner,https://s.yimg.com/ny/api/res/1.2/dUBJSGdIpy02...,2026-04-09T00:00:00,2026-04-12T19:59:59,0.000000,NewsAPI,everything,"""Tommy Fleetwood"" AND (Masters OR Augusta OR g..."
6,https://www.nbcsports.com/golf/news/2026-maste...,Xander Schauffele::2026 Masters final round: S...,Patricia Duffy,The 2026 Masters Tournament concludes Sunday a...,Full Sunday tee times and pairings for the fin...,Xander Schauffele,2026-04-12T01:05:35Z,2026-04-28T21:54:08.137868+00:00,None,NBCSports.com,"2026 Masters final round: Sunday tee times, pa...",Masters Tournament Winner,https://s.yimg.com/ny/api/res/1.2/i4ZeLgBQO3kY...,2026-04-09T00:00:

In [ ]:
def aggregate_news_for_player(player_name, snapshot_time, lookback_hours=24):
    news_collection = get_collection("news_articles")

    snapshot_dt = pd.to_datetime(snapshot_time, utc=True)
    cutoff_dt = snapshot_dt - pd.Timedelta(hours=lookback_hours)

    articles = list(
        news_collection.find(
            {
                "player_name": player_name,
                "published_at": {
                    "$gte": cutoff_dt.isoformat().replace("+00:00", "Z"),
                    "$lte": snapshot_dt.isoformat().replace("+00:00", "Z"),
                },
            },
            {"_id": 0}
        )
    )

    if not articles:
        return {
            "article_count_24h": 0,
            "avg_sentiment_24h": 0.0,
            "distinct_sources_24h": 0,
            "latest_article_age_hours": None,
        }

    sentiments = [
        article.get("sentiment", {}).get("simple_score", 0.0)
        for article in articles
    ]

    source_names = {
        article.get("source_name")
        for article in articles
        if article.get("source_name")
    }

    published_times = [
        pd.to_datetime(article["published_at"], utc=True)
        for article in articles
        if article.get("published_at")
    ]

    latest_article_age_hours = None
    if published_times:
        latest_article_time = max(published_times)
        latest_article_age_hours = (snapshot_dt - latest_article_time).total_seconds() / 3600

    return {
        "article_count_24h": len(articles),
        "avg_sentiment_24h": sum(sentiments) / len(sentiments),
        "distinct_sources_24h": len(source_names),
        "latest_article_age_hours": latest_article_age_hours,
    }

In [ ]:
def build_player_snapshot_docs():
    odds_collection = get_collection("odds_snapshots")
    odds_docs = list(odds_collection.find({}, {"_id": 0}))

    final_docs = []

    for odds_doc in odds_docs:
        player_name = odds_doc["player_name"]
        snapshot_time = odds_doc["snapshot_time_utc"]

        news_features = aggregate_news_for_player(
            player_name=player_name,
            snapshot_time=snapshot_time,
            lookback_hours=24,
        )

        final_doc = {
            "snapshot_time_utc": snapshot_time,
            "requested_snapshot_time_utc": odds_doc.get("requested_snapshot_time_utc"),
            "tournament": odds_doc.get("tournament"),
            "event_id": odds_doc.get("event_id"),
            "event_name": odds_doc.get("event_name"),
            "commence_time": odds_doc.get("commence_time"),
            "player_name": player_name,
            "odds": odds_doc.get("odds", {}),
            "odds_change": odds_doc.get("odds_change", {}),
            "news_features": news_features,
            "feature_vector": {
                "average_american_price": odds_doc.get("odds", {}).get("average_american_price"),
                "raw_implied_prob": odds_doc.get("odds", {}).get("raw_implied_prob"),
                "normalized_implied_prob": odds_doc.get("odds", {}).get("normalized_implied_prob"),
                "sportsbook_quote_count": odds_doc.get("odds", {}).get("sportsbook_quote_count"),
                "odds_change_avg_american_price": odds_doc.get("odds_change", {}).get("average_american_price"),
                "odds_change_raw_implied_prob": odds_doc.get("odds_change", {}).get("raw_implied_prob"),
                "odds_change_normalized_implied_prob": odds_doc.get("odds_change", {}).get("normalized_implied_prob"),
                **news_features,
            },
            "created_from": {
                "odds_collection": "odds_snapshots",
                "news_collection": "news_articles",
            }
        }

        final_docs.append(final_doc)

    return final_docs

In [ ]:
def upsert_player_snapshots(documents):
    if not documents:
        return 0

    collection = get_collection("player_snapshots")
    collection.create_index(
        [("snapshot_time_utc", 1), ("player_name", 1)],
        unique=True
    )

    operations = []

    for doc in documents:
        operations.append(
            UpdateOne(
                {
                    "snapshot_time_utc": doc["snapshot_time_utc"],
                    "player_name": doc["player_name"],
                },
                {"$set": doc},
                upsert=True,
            )
        )

    result = collection.bulk_write(operations, ordered=False)
    return result.upserted_count + result.modified_count

In [ ]:
odds_docs = list(
    get_collection("odds_snapshots").find({}, {"_id": 0})
)

news_docs = list(
    get_collection("news_articles").find({}, {"_id": 0})
)

odds_df = pd.json_normalize(odds_docs)
news_df = pd.json_normalize(news_docs)

print("odds_df shape:", odds_df.shape)
print("news_df shape:", news_df.shape)

odds_df shape: (26157, 18)
news_df shape: (739, 19)


In [ ]:
odds_df["snapshot_time_utc"] = pd.to_datetime(odds_df["snapshot_time_utc"], utc=True)

news_df["published_at"] = pd.to_datetime(news_df["published_at"], utc=True, errors="coerce")
news_df = news_df.dropna(subset=["published_at"]).copy()

if "sentiment.simple_score" not in news_df.columns:
    news_df["sentiment.simple_score"] = 0.0

news_df["source_name"] = news_df["source_name"].fillna("unknown")

In [ ]:
def build_news_feature_table(odds_df, news_df, lookback_hours=24):
    feature_rows = []

    odds_df_sorted = odds_df.sort_values(["player_name", "snapshot_time_utc"]).copy()
    news_df_sorted = news_df.sort_values(["player_name", "published_at"]).copy()

    players = odds_df_sorted["player_name"].dropna().unique()

    for i, player in enumerate(players, start=1):
        player_odds = odds_df_sorted[odds_df_sorted["player_name"] == player].copy()
        player_news = news_df_sorted[news_df_sorted["player_name"] == player].copy()

        if i % 10 == 0:
            print(f"Processing player {i}/{len(players)}: {player}")

        if player_news.empty:
            for _, odds_row in player_odds.iterrows():
                feature_rows.append({
                    "snapshot_time_utc": odds_row["snapshot_time_utc"],
                    "player_name": player,
                    "article_count_24h": 0,
                    "avg_sentiment_24h": 0.0,
                    "distinct_sources_24h": 0,
                    "latest_article_age_hours": None,
                })
            continue

        published_times = player_news["published_at"].tolist()
        sentiments = player_news["sentiment.simple_score"].fillna(0.0).tolist()
        sources = player_news["source_name"].fillna("unknown").tolist()

        for _, odds_row in player_odds.iterrows():
            snapshot_time = odds_row["snapshot_time_utc"]
            cutoff_time = snapshot_time - pd.Timedelta(hours=lookback_hours)

            mask = (player_news["published_at"] >= cutoff_time) & (player_news["published_at"] <= snapshot_time)
            window_news = player_news.loc[mask]

            if window_news.empty:
                feature_rows.append({
                    "snapshot_time_utc": snapshot_time,
                    "player_name": player,
                    "article_count_24h": 0,
                    "avg_sentiment_24h": 0.0,
                    "distinct_sources_24h": 0,
                    "latest_article_age_hours": None,
                })
            else:
                latest_article_time = window_news["published_at"].max()
                latest_article_age_hours = (snapshot_time - latest_article_time).total_seconds() / 3600

                feature_rows.append({
                    "snapshot_time_utc": snapshot_time,
                    "player_name": player,
                    "article_count_24h": int(len(window_news)),
                    "avg_sentiment_24h": float(window_news["sentiment.simple_score"].mean()),
                    "distinct_sources_24h": int(window_news["source_name"].nunique()),
                    "latest_article_age_hours": float(latest_article_age_hours),
                })

    return pd.DataFrame(feature_rows)

In [ ]:
news_features_df = build_news_feature_table(
    odds_df=odds_df,
    news_df=news_df,
    lookback_hours=24
)

print(news_features_df.shape)
news_features_df.head()

Processing player 10/92: Brian Campbell
Processing player 20/92: Christopher Gotterup
Processing player 30/92: Haotong Li
Processing player 40/92: Jon Rahm
Processing player 50/92: Mason Howell
Processing player 60/92: Min Woo Lee
Processing player 70/92: Rasmus Neergaard-Petersen
Processing player 80/92: Sepp Straka
Processing player 90/92: Wyndham Clark
(26157, 6)


,snapshot_time_utc,player_name,article_count_24h,avg_sentiment_24h,distinct_sources_24h,latest_article_age_hours
0,2026-04-08 23:55:38+00:00,Aaron Rai,0,0.000000,0,NaN
1,2026-04-09 00:10:38+00:00,Aaron Rai,0,0.000000,0,NaN
2,2026-04-09 00:25:38+00:00,Aaron Rai,0,0.000000,0,NaN
3,2026-04-09 00:40:38+00:00,Aaron Rai,0,0.000000,0,NaN
4,2026-04-09 00:55:38+00:00,Aaron Rai,1,0.026667,1,0.055


In [ ]:
merged_df = odds_df.merge(
    news_features_df,
    on=["snapshot_time_utc", "player_name"],
    how="left"
)

print(merged_df.shape)
merged_df.head()

(26157, 22)


,player_name,snapshot_time_utc,commence_time,event_id,event_name,requested_snapshot_time_utc,tournament,odds.average_american_price,odds.raw_implied_prob,odds.sportsbook_quote_count,...,source.mode,source.sport_key,source.market,odds_change.average_american_price,odds_change.raw_implied_prob,odds_change.normalized_implied_prob,article_count_24h,avg_sentiment_24h,distinct_sources_24h,latest_article_age_hours
0,Scottie Scheffler,2026-04-08 23:55:38+00:00,2026-04-09T11:40:00Z,b709a17e8bec948af575f4534c598144,Masters Tournament Winner,2026-04-09T00:00:00Z,Masters Tournament Winner,571.666667,0.148883,6,...,historical,golf_masters_tournament_winner,outrights,NaN,NaN,NaN,0,0.0,0,NaN
1,Bryson DeChambeau,2026-04-08 23:55:38+00:00,2026-04-09T11:40:00Z,b709a17e8bec948af575f4534c598144,Masters Tournament Winner,2026-04-09T00:00:00Z,Masters Tournament Winner,1091.666667,0.083916,6,...,historical,golf_masters_tournament_winner,outrights,NaN,NaN,NaN,0,0.0,0,NaN
2,Jon Rahm,2026-04-08 23:55:38+00:00,2026-04-09T11:40:00Z,b709a17e8bec948af575f4534c598144,Masters Tournament Winner,2026-04-09T00:00:00Z,Masters Tournament Winner,1033.333333,0.088235,6,...,historical,golf_masters_tournament_winner,outrights,NaN,NaN,NaN,0,0.0,0,NaN
3,Ludvig Aberg,2026-04-08 23:55:38+00:00,2026-04-09T11:40:00Z,b709a17e8bec948af575f4534c598144,Masters Tournament Winner,2026-04-09T00:00:00Z,Masters Tournament Winner,1583.333333,0.059406,6,...,historical,golf_masters_tournament_winner,outrights,NaN,NaN,NaN,0,0.0,0,NaN
4,Rory McIlroy,2026-04-08 23:55:38+00:00,2026-04-09T11:40:00Z,b709a17e8bec948af575f4534c598144,Masters Tournament Winner,2026-04-09T00:00:00Z,Masters Tournament Winner,1320.833333,0.070381,6,...,historical,golf_masters_tournament_winner,outrights,NaN,NaN,NaN,0,0.0,0,NaN


In [ ]:
merged_df["article_count_24h"] = merged_df["article_count_24h"].fillna(0).astype(int)
merged_df["avg_sentiment_24h"] = merged_df["avg_sentiment_24h"].fillna(0.0)
merged_df["distinct_sources_24h"] = merged_df["distinct_sources_24h"].fillna(0).astype(int)

In [ ]:
def row_to_player_snapshot_doc(row):
    return {
        "snapshot_time_utc": row["snapshot_time_utc"].isoformat().replace("+00:00", "Z"),
        "requested_snapshot_time_utc": row.get("requested_snapshot_time_utc"),
        "tournament": row.get("tournament"),
        "event_id": row.get("event_id"),
        "event_name": row.get("event_name"),
        "commence_time": row.get("commence_time"),
        "player_name": row.get("player_name"),
        "odds": {
            "average_american_price": None if pd.isna(row.get("odds.average_american_price")) else float(row.get("odds.average_american_price")),
            "raw_implied_prob": None if pd.isna(row.get("odds.raw_implied_prob")) else float(row.get("odds.raw_implied_prob")),
            "normalized_implied_prob": None if pd.isna(row.get("odds.normalized_implied_prob")) else float(row.get("odds.normalized_implied_prob")),
            "sportsbook_quote_count": None if pd.isna(row.get("odds.sportsbook_quote_count")) else int(row.get("odds.sportsbook_quote_count")),
        },
        "odds_change": {
            "average_american_price": None if pd.isna(row.get("odds_change.average_american_price")) else float(row.get("odds_change.average_american_price")),
            "raw_implied_prob": None if pd.isna(row.get("odds_change.raw_implied_prob")) else float(row.get("odds_change.raw_implied_prob")),
            "normalized_implied_prob": None if pd.isna(row.get("odds_change.normalized_implied_prob")) else float(row.get("odds_change.normalized_implied_prob")),
        },
        "news_features": {
            "article_count_24h": int(row.get("article_count_24h", 0)),
            "avg_sentiment_24h": float(row.get("avg_sentiment_24h", 0.0)),
            "distinct_sources_24h": int(row.get("distinct_sources_24h", 0)),
            "latest_article_age_hours": None if pd.isna(row.get("latest_article_age_hours")) else float(row.get("latest_article_age_hours")),
        },
        "feature_vector": {
            "average_american_price": None if pd.isna(row.get("odds.average_american_price")) else float(row.get("odds.average_american_price")),
            "raw_implied_prob": None if pd.isna(row.get("odds.raw_implied_prob")) else float(row.get("odds.raw_implied_prob")),
            "normalized_implied_prob": None if pd.isna(row.get("odds.normalized_implied_prob")) else float(row.get("odds.normalized_implied_prob")),
            "sportsbook_quote_count": None if pd.isna(row.get("odds.sportsbook_quote_count")) else int(row.get("odds.sportsbook_quote_count")),
            "odds_change_avg_american_price": None if pd.isna(row.get("odds_change.average_american_price")) else float(row.get("odds_change.average_american_price")),
            "odds_change_raw_implied_prob": None if pd.isna(row.get("odds_change.raw_implied_prob")) else float(row.get("odds_change.raw_implied_prob")),
            "odds_change_normalized_implied_prob": None if pd.isna(row.get("odds_change.normalized_implied_prob")) else float(row.get("odds_change.normalized_implied_prob")),
            "article_count_24h": int(row.get("article_count_24h", 0)),
            "avg_sentiment_24h": float(row.get("avg_sentiment_24h", 0.0)),
            "distinct_sources_24h": int(row.get("distinct_sources_24h", 0)),
            "latest_article_age_hours": None if pd.isna(row.get("latest_article_age_hours")) else float(row.get("latest_article_age_hours")),
        },
        "created_from": {
            "odds_collection": "odds_snapshots",
            "news_collection": "news_articles",
        }
    }

In [ ]:
player_snapshot_docs = [row_to_player_snapshot_doc(row) for _, row in merged_df.iterrows()]

print("Player snapshot docs built:", len(player_snapshot_docs))

Player snapshot docs built: 26157


In [ ]:
def upsert_player_snapshots(documents):
    if not documents:
        return 0

    collection = get_collection("player_snapshots")
    collection.create_index(
        [("snapshot_time_utc", 1), ("player_name", 1)],
        unique=True
    )

    operations = []

    for doc in documents:
        operations.append(
            UpdateOne(
                {
                    "snapshot_time_utc": doc["snapshot_time_utc"],
                    "player_name": doc["player_name"],
                },
                {"$set": doc},
                upsert=True,
            )
        )

    result = collection.bulk_write(operations, ordered=False)
    return result.upserted_count + result.modified_count

In [ ]:
writes = upsert_player_snapshots(player_snapshot_docs)
print("Mongo writes:", writes)

Mongo writes: 26157


In [ ]:
final_docs = list(
    get_collection("player_snapshots").find({}, {"_id": 0}).limit(10)
)

final_df = pd.json_normalize(final_docs)
print(final_df.shape)
final_df.head()

(10, 31)


,player_name,snapshot_time_utc,commence_time,event_id,event_name,requested_snapshot_time_utc,tournament,created_from.odds_collection,created_from.news_collection,feature_vector.average_american_price,...,news_features.avg_sentiment_24h,news_features.distinct_sources_24h,news_features.latest_article_age_hours,odds.average_american_price,odds.raw_implied_prob,odds.normalized_implied_prob,odds.sportsbook_quote_count,odds_change.average_american_price,odds_change.raw_implied_prob,odds_change.normalized_implied_prob
0,Scottie Scheffler,2026-04-08T23:55:38Z,2026-04-09T11:40:00Z,b709a17e8bec948af575f4534c598144,Masters Tournament Winner,2026-04-09T00:00:00Z,Masters Tournament Winner,odds_snapshots,news_articles,571.666667,...,0.0,0,None,571.666667,0.148883,0.110615,6,None,None,None
1,Bryson DeChambeau,2026-04-08T23:55:38Z,2026-04-09T11:40:00Z,b709a17e8bec948af575f4534c598144,Masters Tournament Winner,2026-04-09T00:00:00Z,Masters Tournament Winner,odds_snapshots,news_articles,1091.666667,...,0.0,0,None,1091.666667,0.083916,0.062346,6,None,None,None
2,Jon Rahm,2026-04-08T23:55:38Z,2026-04-09T11:40:00Z,b709a17e8bec948af575f4534c598144,Masters Tournament Winner,2026-04-09T00:00:00Z,Masters Tournament Winner,odds_snapshots,news_articles,1033.333333,...,0.0,0,None,1033.333333,0.088235,0.065555,6,None,None,None
3,Ludvig Aberg,2026-04-08T23:55:38Z,2026-04-09T11:40:00Z,b709a17e8bec948af575f4534c598144,Masters Tournament Winner,2026-04-09T00:00:00Z,Masters Tournament Winner,odds_snapshots,news_articles,1583.333333,...,0.0,0,None,1583.333333,0.059406,0.044136,6,None,None,None
4,Rory McIlroy,2026-04-08T23:55:38Z,2026-04-09T11:40:00Z,b709a17e8bec948af575f4534c598144,Masters Tournament Winner,2026-04-09T00:00:00Z,Masters Tournament Winner,odds_snapshots,news_articles,1320.833333,...,0.0,0,None,1320.833333,0.070381,0.052291,6,None,None,None


In [ ]:
print("player_snapshots count:", get_collection("player_snapshots").count_documents({}))

In [ ]:
final_docs = list(
    get_collection("player_snapshots").find({}, {"_id": 0}).limit(10)
)

final_df = pd.json_normalize(final_docs)
print(final_df.shape)
final_df.head()

(10, 31)


,player_name,snapshot_time_utc,commence_time,event_id,event_name,requested_snapshot_time_utc,tournament,created_from.odds_collection,created_from.news_collection,feature_vector.average_american_price,...,news_features.avg_sentiment_24h,news_features.distinct_sources_24h,news_features.latest_article_age_hours,odds.average_american_price,odds.raw_implied_prob,odds.normalized_implied_prob,odds.sportsbook_quote_count,odds_change.average_american_price,odds_change.raw_implied_prob,odds_change.normalized_implied_prob
0,Scottie Scheffler,2026-04-08T23:55:38Z,2026-04-09T11:40:00Z,b709a17e8bec948af575f4534c598144,Masters Tournament Winner,2026-04-09T00:00:00Z,Masters Tournament Winner,odds_snapshots,news_articles,571.666667,...,0.0,0,None,571.666667,0.148883,0.110615,6,None,None,None
1,Bryson DeChambeau,2026-04-08T23:55:38Z,2026-04-09T11:40:00Z,b709a17e8bec948af575f4534c598144,Masters Tournament Winner,2026-04-09T00:00:00Z,Masters Tournament Winner,odds_snapshots,news_articles,1091.666667,...,0.0,0,None,1091.666667,0.083916,0.062346,6,None,None,None
2,Jon Rahm,2026-04-08T23:55:38Z,2026-04-09T11:40:00Z,b709a17e8bec948af575f4534c598144,Masters Tournament Winner,2026-04-09T00:00:00Z,Masters Tournament Winner,odds_snapshots,news_articles,1033.333333,...,0.0,0,None,1033.333333,0.088235,0.065555,6,None,None,None
3,Ludvig Aberg,2026-04-08T23:55:38Z,2026-04-09T11:40:00Z,b709a17e8bec948af575f4534c598144,Masters Tournament Winner,2026-04-09T00:00:00Z,Masters Tournament Winner,odds_snapshots,news_articles,1583.333333,...,0.0,0,None,1583.333333,0.059406,0.044136,6,None,None,None
4,Rory McIlroy,2026-04-08T23:55:38Z,2026-04-09T11:40:00Z,b709a17e8bec948af575f4534c598144,Masters Tournament Winner,2026-04-09T00:00:00Z,Masters Tournament Winner,odds_snapshots,news_articles,1320.833333,...,0.0,0,None,1320.833333,0.070381,0.052291,6,None,None,None


In [ ]:
print("odds_snapshots:", get_collection("odds_snapshots").count_documents({}))
print("news_articles:", get_collection("news_articles").count_documents({}))
print("player_snapshots:", get_collection("player_snapshots").count_documents({}))

odds_snapshots: 26157
news_articles: 739
player_snapshots: 26157


In [ ]:
change_check = list(
    get_collection("player_snapshots").find(
        {"odds_change.normalized_implied_prob": {"$ne": None}},
        {
            "_id": 0,
            "player_name": 1,
            "snapshot_time_utc": 1,
            "odds_change": 1
        }
    ).limit(5)
)

pd.json_normalize(change_check)

,snapshot_time_utc,player_name,odds_change.average_american_price,odds_change.raw_implied_prob,odds_change.normalized_implied_prob
0,2026-04-09T00:10:38Z,Scottie Scheffler,0.0,0.0,-0.046391
1,2026-04-09T00:10:38Z,Bryson DeChambeau,0.0,0.0,-0.026148
2,2026-04-09T00:10:38Z,Jon Rahm,0.0,0.0,-0.027493
3,2026-04-09T00:10:38Z,Ludvig Aberg,0.0,0.0,-0.018510
4,2026-04-09T00:10:38Z,Rory McIlroy,0.0,0.0,-0.021930


In [ ]:
news_check = list(
    get_collection("player_snapshots").find(
        {},
        {
            "_id": 0,
            "player_name": 1,
            "snapshot_time_utc": 1,
            "news_features": 1
        }
    ).limit(10)
)

pd.json_normalize(news_check)

,player_name,snapshot_time_utc,news_features.article_count_24h,news_features.avg_sentiment_24h,news_features.distinct_sources_24h,news_features.latest_article_age_hours
0,Scottie Scheffler,2026-04-08T23:55:38Z,0,0.0,0,None
1,Bryson DeChambeau,2026-04-08T23:55:38Z,0,0.0,0,None
2,Jon Rahm,2026-04-08T23:55:38Z,0,0.0,0,None
3,Ludvig Aberg,2026-04-08T23:55:38Z,0,0.0,0,None
4,Rory McIlroy,2026-04-08T23:55:38Z,0,0.0,0,None
5,Xander Schauffele,2026-04-08T23:55:38Z,0,0.0,0,None
6,Cameron Young,2026-04-08T23:55:38Z,0,0.0,0,None
7,Matthew Fitzpatrick,2026-04-08T23:55:38Z,0,0.0,0,None
8,Tommy Fleetwood,2026-04-08T23:55:38Z,0,0.0,0,None
9,Hideki Matsuyama,2026-04-08T23:55:38Z,0,0.0,0,None


In [ ]:
full_df = pd.json_normalize(
    list(get_collection("player_snapshots").find({}, {"_id": 0}))
)

# Check how many non-zero changes exist
print(
    "Non-zero odds changes:",
    (full_df["odds_change.normalized_implied_prob"] != 0).sum()
)

Non-zero odds changes: 19627


In [ ]:
print("odds_snapshots:", get_collection("odds_snapshots").count_documents({}))
print("news_articles:", get_collection("news_articles").count_documents({}))
print("player_snapshots:", get_collection("player_snapshots").count_documents({}))

full_df = pd.json_normalize(
    list(get_collection("player_snapshots").find({}, {"_id": 0}))
)

print("\nShape:", full_df.shape)
print("Unique players:", full_df["player_name"].nunique())
print("Unique timestamps:", full_df["snapshot_time_utc"].nunique())
print("Rows with news > 0:", (full_df["news_features.article_count_24h"] > 0).sum())

full_df.head()

odds_snapshots: 26157
news_articles: 739
player_snapshots: 26157

Shape: (26157, 31)
Unique players: 92
Unique timestamps: 365
Rows with news > 0: 10102


,player_name,snapshot_time_utc,commence_time,event_id,event_name,requested_snapshot_time_utc,tournament,created_from.odds_collection,created_from.news_collection,feature_vector.average_american_price,...,news_features.avg_sentiment_24h,news_features.distinct_sources_24h,news_features.latest_article_age_hours,odds.average_american_price,odds.raw_implied_prob,odds.normalized_implied_prob,odds.sportsbook_quote_count,odds_change.average_american_price,odds_change.raw_implied_prob,odds_change.normalized_implied_prob
0,Scottie Scheffler,2026-04-08T23:55:38Z,2026-04-09T11:40:00Z,b709a17e8bec948af575f4534c598144,Masters Tournament Winner,2026-04-09T00:00:00Z,Masters Tournament Winner,odds_snapshots,news_articles,571.666667,...,0.0,0,NaN,571.666667,0.148883,0.110615,6,NaN,NaN,NaN
1,Bryson DeChambeau,2026-04-08T23:55:38Z,2026-04-09T11:40:00Z,b709a17e8bec948af575f4534c598144,Masters Tournament Winner,2026-04-09T00:00:00Z,Masters Tournament Winner,odds_snapshots,news_articles,1091.666667,...,0.0,0,NaN,1091.666667,0.083916,0.062346,6,NaN,NaN,NaN
2,Jon Rahm,2026-04-08T23:55:38Z,2026-04-09T11:40:00Z,b709a17e8bec948af575f4534c598144,Masters Tournament Winner,2026-04-09T00:00:00Z,Masters Tournament Winner,odds_snapshots,news_articles,1033.333333,...,0.0,0,NaN,1033.333333,0.088235,0.065555,6,NaN,NaN,NaN
3,Ludvig Aberg,2026-04-08T23:55:38Z,2026-04-09T11:40:00Z,b709a17e8bec948af575f4534c598144,Masters Tournament Winner,2026-04-09T00:00:00Z,Masters Tournament Winner,odds_snapshots,news_articles,1583.333333,...,0.0,0,NaN,1583.333333,0.059406,0.044136,6,NaN,NaN,NaN
4,Rory McIlroy,2026-04-08T23:55:38Z,2026-04-09T11:40:00Z,b709a17e8bec948af575f4534c598144,Masters Tournament Winner,2026-04-09T00:00:00Z,Masters Tournament Winner,odds_snapshots,news_articles,1320.833333,...,0.0,0,NaN,1320.833333,0.070381,0.052291,6,NaN,NaN,NaN


In [ ]:
full_df = full_df.sort_values(["player_name", "snapshot_time_utc"])

full_df["next_implied_prob"] = (
    full_df.groupby("player_name")["odds.normalized_implied_prob"].shift(-1)
)

full_df["target_prob_up_next"] = (
    full_df["next_implied_prob"] > full_df["odds.normalized_implied_prob"]
).astype(int)

In [ ]:
pip install pymongo

In [ ]:
import os
import pandas as pd
from pymongo import MongoClient
from dotenv import load_dotenv

load_dotenv()

client = MongoClient(MONGO_URI)

db = client["project2"]
collection = db["player_snapshots"]

docs = list(collection.find({}))

df = pd.json_normalize(docs)

df.head()

""


In [ ]:
client.list_database_names()

['masters_project', 'sample_mflix', 'synthea_fhir', 'admin', 'local']

In [ ]:
db = client["masters_project"]

In [ ]:
db.list_collection_names()

['news_articles', 'player_snapshots', 'odds_snapshots']

In [ ]:
for coll_name in db.list_collection_names():
    count = db[coll_name].count_documents({})
    print(coll_name, count)

news_articles 739
player_snapshots 35476
odds_snapshots 35476


In [ ]:
odds_df = pd.json_normalize(list(db["odds_snapshots"].find({})))
news_df = pd.json_normalize(list(db["news_articles"].find({})))
player_df = pd.json_normalize(list(db["player_snapshots"].find({})))

print("Odds:", odds_df.shape)
print("News:", news_df.shape)
print("Player snapshots:", player_df.shape)

Odds: (35476, 19)
News: (739, 20)
Player snapshots: (35476, 32)


In [ ]:
df = player_df.copy()

df["_id"] = df["_id"].astype(str)
df["snapshot_time_utc"] = pd.to_datetime(df["snapshot_time_utc"])

print(df.shape)
df.head()

(35476, 32)


,_id,snapshot_time_utc,player_name,commence_time,event_id,event_name,requested_snapshot_time_utc,tournament,created_from.odds_collection,created_from.news_collection,...,news_features.avg_sentiment_24h,news_features.distinct_sources_24h,news_features.latest_article_age_hours,odds.average_american_price,odds.raw_implied_prob,odds.normalized_implied_prob,odds.sportsbook_quote_count,odds_change.average_american_price,odds_change.raw_implied_prob,odds_change.normalized_implied_prob
0,69e7c4cb30f8268f4443c019,2026-04-05 23:55:38+00:00,Scottie Scheffler,2026-04-12T21:45:00Z,b709a17e8bec948af575f4534c598144,Masters Tournament Winner,2026-04-06T00:00:00Z,Masters Tournament Winner,odds_snapshots,news_articles,...,0.0,0,NaN,515.00,0.162602,0.116660,4,NaN,NaN,NaN
1,69e7c4cb30f8268f4443c01a,2026-04-05 23:55:38+00:00,Bryson DeChambeau,2026-04-12T21:45:00Z,b709a17e8bec948af575f4534c598144,Masters Tournament Winner,2026-04-06T00:00:00Z,Masters Tournament Winner,odds_snapshots,news_articles,...,0.0,0,NaN,1050.00,0.086957,0.062388,4,NaN,NaN,NaN
2,69e7c4cb30f8268f4443c01b,2026-04-05 23:55:38+00:00,Rory McIlroy,2026-04-12T21:45:00Z,b709a17e8bec948af575f4534c598144,Masters Tournament Winner,2026-04-06T00:00:00Z,Masters Tournament Winner,odds_snapshots,news_articles,...,0.0,0,NaN,1106.25,0.082902,0.059479,4,NaN,NaN,NaN
3,69e7c4cb30f8268f4443c01d,2026-04-05 23:55:38+00:00,Ludvig Aberg,2026-04-12T21:45:00Z,b709a17e8bec948af575f4534c598144,Masters Tournament Winner,2026-04-06T00:00:00Z,Masters Tournament Winner,odds_snapshots,news_articles,...,0.0,0,NaN,1537.50,0.061069,0.043814,4,NaN,NaN,NaN
4,69e7c4cb30f8268f4443c01e,2026-04-05 23:55:38+00:00,Xander Schauffele,2026-04-12T21:45:00Z,b709a17e8bec948af575f4534c598144,Masters Tournament Winner,2026-04-06T00:00:00Z,Masters Tournament Winner,odds_snapshots,news_articles,...,0.0,0,NaN,1662.50,0.056738,0.040707,4,NaN,NaN,NaN


In [ ]:
df = df.sort_values(["player_name", "snapshot_time_utc"])

df["next_normalized_prob"] = (
    df.groupby("player_name")["odds.normalized_implied_prob"].shift(-1)
)

df["target_prob_up_next"] = (
    df["next_normalized_prob"] > df["odds.normalized_implied_prob"]
).astype(int)